In [1]:
from importlib.metadata import version

print("torch version: ", version("torch"))
print("transformers version: ", version("transformers"))
print("datasets version: ", version("datasets"))

torch version:  2.8.0+cu129
transformers version:  4.57.6
datasets version:  4.5.0


In [2]:
import os

# 切到 train_llm（notebook 所在目录）
os.chdir(r"C:\Users\1\Desktop\learn-torch\Step4 LLM预训练\train_llm")

# 验证
print(os.getcwd())
print(os.path.exists("tokenizers/Qwen/Qwen3-0.6B/config.json"))  # 应输出 True

C:\Users\1\Desktop\learn-torch\Step4 LLM预训练\train_llm
True


In [3]:
import torch
import transformers
import datasets
import json
import math


D:\Anaconda\envs\pytorch\lib\ssl.py:569: UserWarning: Failed to load certificates from Windows store 'CA': [ASN1: NOT_ENOUGH_DATA] not enough data (_ssl.c:4192)
  warnings.warn(f"Failed to load certificates from Windows store '{storename}': {e}")
D:\Anaconda\envs\pytorch\lib\ssl.py:569: UserWarning: Failed to load certificates from Windows store 'ROOT': [ASN1: NOT_ENOUGH_DATA] not enough data (_ssl.c:4192)
  warnings.warn(f"Failed to load certificates from Windows store '{storename}': {e}")


In [4]:
# 加载分词器
tokenizer = transformers.AutoTokenizer.from_pretrained("C:\\Users\\1\\Desktop\\learn-torch\\Step4 LLM预训练\\train_llm\\tokenizers\\merged_tokenizer")

In [5]:
tokenizer.vocab_size


58703

In [6]:
#重新定义模型结构参数
context_length = 512  #上下文长度
config = transformers.AutoConfig.from_pretrained(
    "tokenizers/Qwen/Qwen3-0.6B/",
    bos_token_id = 1,
    eos_token_id = 2,
    hidden_size = 512,
    intermediate_size = 1536,
    max_position_embedding = 8192,
    num_attention_heads = 8,
    num_key_value_heads=4,
    num_hidden_layers=16,
    rope_theta=10000,
    vocab_size=58703,
    n_ctx=context_length
)


In [7]:
print(config)

Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 512,
  "initializer_range": 0.02,
  "intermediate_size": 1536,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 40960,
  "max_window_layer

In [8]:
# embedding层的参数
#150000 * 1024 * 2 / 1e6
58703 * 512 * 2 / 1e6

60.111872

In [9]:
model = transformers.Qwen3ForCausalLM(config)
print("Model Summary:")
print(model)

Model Summary:
Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(58703, 512)
    (layers): ModuleList(
      (0-15): 16 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=512, out_features=1024, bias=False)
          (k_proj): Linear(in_features=512, out_features=512, bias=False)
          (v_proj): Linear(in_features=512, out_features=512, bias=False)
          (o_proj): Linear(in_features=1024, out_features=512, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=512, out_features=1536, bias=False)
          (up_proj): Linear(in_features=512, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=512, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((512,), eps=1e-06)
        (post_attention_la

In [10]:
# 打印所有参数的shape和名称
print("Model Parameters and Shape:")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")

Model Parameters and Shape:
model.embed_tokens.weight: torch.Size([58703, 512])
model.layers.0.self_attn.q_proj.weight: torch.Size([1024, 512])
model.layers.0.self_attn.k_proj.weight: torch.Size([512, 512])
model.layers.0.self_attn.v_proj.weight: torch.Size([512, 512])
model.layers.0.self_attn.o_proj.weight: torch.Size([512, 1024])
model.layers.0.self_attn.q_norm.weight: torch.Size([128])
model.layers.0.self_attn.k_norm.weight: torch.Size([128])
model.layers.0.mlp.gate_proj.weight: torch.Size([1536, 512])
model.layers.0.mlp.up_proj.weight: torch.Size([1536, 512])
model.layers.0.mlp.down_proj.weight: torch.Size([512, 1536])
model.layers.0.input_layernorm.weight: torch.Size([512])
model.layers.0.post_attention_layernorm.weight: torch.Size([512])
model.layers.1.self_attn.q_proj.weight: torch.Size([1024, 512])
model.layers.1.self_attn.k_proj.weight: torch.Size([512, 512])
model.layers.1.self_attn.v_proj.weight: torch.Size([512, 512])
model.layers.1.self_attn.o_proj.weight: torch.Size([512,

In [11]:
# 加载预训练数据
raw_datasets = datasets.load_dataset(
    "json", data_files="data/processed_dataset_512_clean_230w.jsonl"
)
print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2299107
    })
})


In [12]:
raw_datasets = raw_datasets["train"].train_test_split(test_size=0.005, seed=42)
print("dataset info:")
print(raw_datasets)

dataset info:
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2287611
    })
    test: Dataset({
        features: ['text'],
        num_rows: 11496
    })
})


In [13]:
context_length = 512

def tokenize(element, tokenizer=tokenizer, context_length=context_length):
    outputs = tokenizer(
        element['text'],
        add_special_tokens=False
    )
    input_ids_list = outputs['input_ids']
    new_input_ids_list, new_attn_mask_list = [], []
    for input_ids in input_ids_list:
        input_ids_eos = input_ids[:context_length-1] + [tokenizer.eos_token_id]
        new_input_ids_list.append(input_ids_eos)
        new_attn_mask_list.append([1] * len(input_ids_eos))
    return {
        "input_ids": new_input_ids_list,
        "attention_mask": new_attn_mask_list
    }

In [14]:
raw_datasets["train"][0]

{'text': '根据所选的人物描述,生成一段描述该人物特征的段落。\n\n一个聪明、勇敢、重视家庭的幽默家庭男人。他是个体育爱好者,喜欢打篮球和高尔夫球。这位男人是个典型的家庭男人,十分聪明、勇敢,是他家庭中的支柱。他热爱运动,从事体育活动已成为他生活中不可或缺的一部分。篮球和高尔夫球是他最热爱的运动,他经常在周末和家人或朋友一起打球。除此之外,他的幽默感也是他的另一个特征,常常给予周围人一些诙谐有趣的说话或行为。'}

In [15]:

# tokenizer原始文本数据
tokenized_datasets = raw_datasets.map(
    tokenize, batched=True, num_proc=16, remove_columns=raw_datasets["train"].column_names
)

In [16]:
print("tokenized dataset info:")
print(tokenized_datasets)

tokenized dataset info:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2287611
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 11496
    })
})


In [17]:
input_examples = [
    {"input_ids": [5714, 876, 272, 2], 'attention_mask': [1, 1, 1, 1]},
    {'input_ids': [22, 1491, 1664, 4532, 292, 4571, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}
]
# input: a, b, c, d
# label: b, c, d, <eos>

# 测试data_collator
data_collator = transformers.DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(data_collator(input_examples))

{'input_ids': tensor([[5714,  876,  272,    2,    0,    0,    0],
        [  22, 1491, 1664, 4532,  292, 4571,    2]]), 'attention_mask': tensor([[1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1]]), 'labels': tensor([[5714,  876,  272,    2, -100, -100, -100],
        [  22, 1491, 1664, 4532,  292, 4571,    2]])}


In [20]:
# 训练参数

args = transformers.TrainingArguments(
    output_dir='saved/',
    per_device_train_batch_size=16, # 每个GPU的训练batch数
    per_device_eval_batch_size=16, # 每个GPU的验证batch数
    gradient_accumulation_steps=8, # 梯度的累积步数
    eval_strategy='steps',
    eval_steps=10,
    logging_steps=5,
    num_train_epochs=2, # 训练的epochs数
    weight_decay=0.1,   # weight decay的比率
    optim='adamw_torch', # 优化器选择AdamW
    warmup_ratio=0.1,   # warmup的比率
    lr_scheduler_type='cosine', # 学习率的衰减策略, [0, T/4]
    learning_rate=3e-4, # 学习率，[1e-4, 5e-5]
    save_steps=500,
    save_total_limit=2, # 最大保存5个ckpt
    bf16=True, # 开启bf16训练，对于Amper架构以下的显卡建议替换为fp16
)
print("Train Args:")
print(args)

Train Args:
TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=10,
eval_strategy=steps,
eval_use_gather_object=False,
fp16=Fa